In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
# load data

sales = pd.read_csv('cozio_sales_ALL.csv')
CPI_US = pd.read_csv('./Data/Economic_Data/US_CPI_1982=100_noseasadj.csv', index_col='observation_date', parse_dates=True)
CPI_UK = pd.read_csv('./Data/Economic_Data/UK_CPI_2015=100.csv', index_col='observation_date', parse_dates=True)
CPI_EU = pd.read_csv('./Data/Economic_Data/EUR_CPI_2015=100.csv', index_col='observation_date', parse_dates=True)
FX_GBP = pd.read_csv('./Data/Economic_Data/GBP_USD.csv', index_col='Date', parse_dates=True, dayfirst=True)
FX_GBP = 1/2 * (FX_GBP.ffill() + FX_GBP.bfill()) # fill NaNs with average value of preceding and next 
FX_EUR = pd.read_csv('./Data/Economic_Data/EUR_USD.csv', sep=';', index_col='Date', parse_dates=True)
FX_EUR['Value'] = FX_EUR['Value'].astype(str).str.replace(',', '.', regex=False) # separation of digits with '.', not ','
FX_EUR['Value'] = FX_EUR['Value'].astype(float)

In [3]:
# load Financial Time Series
US10 = pd.read_csv('./Data/Economic_Data/10Y_Yield_US_percent.csv', index_col='observation_date', parse_dates=True)
US10 = 1/2 * (US10.bfill() + US10.ffill()) # fill with average from previous + next
US10.rename(columns = {'DGS10': '10y_yield_pc'}, inplace=True)

# these two have previously been preprocessed and inflation-adjusted
Gold = pd.read_csv('./Data/Economic_Data/Gold_real.csv', index_col='date', parse_dates=True)

SP = pd.read_csv('./Data/Economic_Data/SP500_index_real.csv', index_col='Date', parse_dates=True).drop(columns='Close')

## Description
- This notebook converts the Tarisio date data into a pandas-interpretable datetime-index.
- It further adjusts the sales prices by
  1. CPI adjusting the local currency, in which the transaction took place,
  2. then converting the local currency into USD.
  3. calculats the _real_adjusted_price_ in USD.
- There is more data for future features, in particular the Dow Jones stock index and 10Y US govt bond rates
* I think this data pre-processing should eventually be outsourced into a pure Python script (or at leat into a Python script, fro which the functions below may be called as from a package) as it is bad taste to crowd Jupyter notebooks, with such big codes. Jupyter notebooks are not for scripting but for visualization and exploration, etc. 

In [4]:
def year_month_column_maker(df, day=False):
    if df.index.dtype in ['<M8[ns]', 'datetime64[ns]']:
        df['Year'] = df.index.year
        df['Month'] = df.index.month
        if day == True:
            df['Day'] = df.index.day
        return df
    else:
        print('Index datatype must be <M8[ns].')

def CPI_preprocess(df):
    #df['observation_date'] = pd.to_datetime(df['observation_date'])
    #df.set_index('observation_date', inplace=True, drop=True)

    df.columns = ['CPI'] # consistent column naming
    # fill NaNs with avg between previous and subsequent value
    if df.CPI.isna().sum() > 0:
        df.loc[df.CPI.isna()] = (0.5 * (df.CPI.ffill() + df.CPI.bfill())).loc[df.CPI.isna()] 
    
    df = year_month_column_maker(df)

    # rebase CPI, such that current month has a multplier of 1, i.e CPI_t -> CPI_t / CPI_last
    df['CPI'] = df['CPI'] / df['CPI'].iloc[-1]

    return df

def currency_preprocess(df, currency, dayfirst=False):
    #df['Date'] = pd.to_datetime(df['Date'], dayfirst=dayfirst)
    #df.set_index('Date', inplace=True, drop=True)
    
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

    if currency == 'gbp':
        df = df.rename(columns={'Value': 'gbp_usd'})
    if currency == 'eur':
        df = df.rename(columns={'Value': 'eur_usd'})

    df = year_month_column_maker(df, day=True) # add year, month, day column to adjust prices later

    
    return df

def attach_timeseries_features(sales, ts_df, cols=None):
    """
    Attach latest available values of ts_df to each sale date.

    sales  : DataFrame with DatetimeIndex
    ts_df  : DataFrame with DatetimeIndex
    cols   : columns to merge (default = all)
    """

    if cols:
        ts_df = ts_df[cols]

    return pd.merge_asof(sales.sort_index(), ts_df.sort_index(), left_index=True, right_index=True, direction="backward")

def z(s, w=252):
    '''
    Rolling Z-score normalisation (zero mean, unit variance) over time window of length w.
    '''
    return (s - s.rolling(w).mean()) / s.rolling(w).std()

In [5]:
sales

,maker_id,maker_name,type,city_maker,auction_house,sale_date,lot,usd,gbp,eur,bold_currency
0,2919,"Achner, Michael",Violin,Wallgau,Tarisio,"Feb 20, 2010",354,4200.0,2721.0,3107.0,usd
1,2919,"Achner, Michael",Violin,Mittenwald,Bongartz's,"Apr 27, 1987",145,2838.0,1702.0,NaN,NaN
2,2611,"Achner, Philip",Violin,Mittenwald,Tarisio,"May 17, 2018",123,24000.0,17759.0,20330.0,usd
3,2611,"Achner, Philip",Violin,NaN,Bongartz's,"Nov 15, 2008",138,3884.0,2613.0,3064.0,NaN
4,2611,"Achner, Philip",Violin,Mittenwald,Sotheby's,"Mar 27, 1990",16,2146.0,1320.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
57164,844,"Zygmuntowicz, Samuel",Violin,"Brooklyn, NY",Tarisio,"Apr 27, 2012",311,108000.0,66560.0,81639.0,usd
57165,844,"Zygmuntowicz, Samuel",Violin,"New York, NY",Brompton's,"Dec 14, 2009",-,15634.0,9600.0,10674.0,gbp
57166,844,"Zygmuntowicz, Samuel",Violin,NaN,Brompton's,"Dec 14, 2009",1,15634.0,9600.0,10674.0,gbp
57167,844,"Zygmuntowicz, Samuel",Violin,"Brooklyn, NY",Tarisio,"May 8, 2003",144,130000.0,81078.0,113686.0,usd


In [6]:
CPI_UK

,GBRCPIALLMINMEI
observation_date,
1955-01-01,4.859513
1955-02-01,4.859513
1955-03-01,4.859513
1955-04-01,4.885972
1955-05-01,4.877152
...,...
2024-11-01,134.600000
2024-12-01,135.100000
2025-01-01,135.100000


In [5]:
def make_datetime(s):
    '''s is a date string of the form "Mon d, yyyy" or "Mon dd, yyyy"
    here Mon is the 3 letter 'word' abbreviation of each month (e.g. 'Feb' for february),
    d, dd are numbers amd yyyy are numbers (where the number of same letter strings encodes its length)
    '''
    month_literal_number_converter = {
        'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
        'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
    }
    month = month_literal_number_converter[s[:3]]
    year = s[-4:]
    day = s[4:6].replace(',', '')
    if int(day)<10:
        day = '0' + day
    #print(day)
    #print('m', month)
    try:
        return pd.to_datetime(day+'/'+month+'/'+year, format="%d/%m/%Y")
    except:
        return pd.NA # very few false dates cannot be converted -> drop those later as NaNs


def adjust_prices(df, CPI_US, CPI_UK, CPI_EU, FX_GBP, FX_EUR):

    CPI_US = CPI_US.rename(columns={'CPI': 'cpi_usd'})
    CPI_UK = CPI_UK.rename(columns={'CPI': 'cpi_gbp'})
    CPI_EU = CPI_EU.rename(columns={'CPI': 'cpi_eur'})

    '''#  merge CPI columns into sales df
    df = df.merge(CPI_US, on=['Year','Month'], how='left')
    df = df.merge(CPI_UK, on=['Year','Month'], how='left')
    df = df.merge(CPI_EU, on=['Year','Month'], how='left')

    # merge FX columns into sales df
    df = df.merge(FX_GBP, on=['Year','Month', 'Day'], how='left')
    df = df.merge(FX_EUR, on=['Year','Month', 'Day'], how='left')'''
    df = attach_timeseries_features(df, CPI_US, ['cpi_usd'])
    df = attach_timeseries_features(df, CPI_UK, ['cpi_gbp'])
    df = attach_timeseries_features(df, CPI_EU, ['cpi_eur'])

    df = attach_timeseries_features(df, FX_GBP, ['gbp_usd'])
    df = attach_timeseries_features(df, FX_EUR, ['eur_usd'])

    

    # inflation adjust local currency
    df.loc[df.bold_currency == 'usd', 'usd'] /= df.loc[df.bold_currency == 'usd', 'cpi_usd']
    df.loc[df.bold_currency == 'gbp', 'gbp'] /= df.loc[df.bold_currency == 'gbp', 'cpi_gbp']
    df.loc[df.bold_currency == 'eur', 'eur'] /= df.loc[df.bold_currency == 'eur', 'cpi_eur']

    # USD conversion for EUR and GBP
    df['price_usd_real'] = np.nan

    mask_usd = df.bold_currency == 'usd'
    mask_gbp = df.bold_currency == 'gbp'
    mask_eur = df.bold_currency == 'eur'
    
    df.loc[mask_usd, 'price_usd_real'] = df.loc[mask_usd, 'usd']
    df.loc[mask_gbp, 'price_usd_real'] = df.loc[mask_gbp, 'gbp'] * df.loc[mask_gbp, 'gbp_usd']
    df.loc[mask_eur, 'price_usd_real'] = df.loc[mask_eur, 'eur'] * df.loc[mask_eur, 'eur_usd']


    return df


def sales_preprocessing(df, CPI_US, CPI_UK, CPI_EU, FX_GBP, FX_EUR):

    df.dropna(subset=['sale_date'], inplace=True)
    df.index = df['sale_date']
    df.sort_index(inplace=True)

    df = year_month_column_maker(df, day=True) # add year, month, day column to adjust prices later

    df = adjust_prices(df, CPI_US, CPI_UK, CPI_EU, FX_GBP, FX_EUR)

    return df
    

In [6]:
# convert to sale date to datetime index
sales['sale_date'] = sales['sale_date'].map(make_datetime) # convert date to pd datetime
sales.dropna(subset=['sale_date'], inplace=True)
sales.index = sales['sale_date']


# make right-formatted CPI series
CPI_US = CPI_preprocess(CPI_US)
CPI_UK = CPI_preprocess(CPI_UK)
CPI_EU = CPI_preprocess(CPI_EU)

# make right-formatted currency conversion series
FX_EUR = currency_preprocess(FX_EUR, 'eur')
FX_GBP = currency_preprocess(FX_GBP, 'gbp', dayfirst=True)


sales = sales_preprocessing(sales, CPI_US, CPI_UK, CPI_EU, FX_GBP, FX_EUR)

In [7]:
# filters out all those that have no adjusted price, eg those that have bold_currency='nan'
sales.dropna(subset=['price_usd_real'], inplace=True)
sales = sales['1975':] # from 1975 onwards we have most related financial time series (additional features)

In [10]:
#sales.to_csv('price_adjusted_data.csv')

In [11]:
sales

,maker_id,maker_name,type,city_maker,auction_house,sale_date,lot,usd,gbp,eur,bold_currency,Year,Month,Day,cpi_usd,cpi_gbp,cpi_eur,gbp_usd,eur_usd,price_usd_real
sale_date,,,,,,,,,,,,,,,,,,,,
1975-02-13,809,"Vuillaume, Jean-Baptiste",Violin,Paris,Sotheby's,1975-02-13,220,7885.000000,32015.631081,3914.000000,gbp,1975,2,13,0.161413,0.103075,NaN,2.3901,NaN,76520.559848
1975-02-13,551,"Platner, Michele",Violin,Rome,Sotheby's,1975-02-13,210,4540.000000,18433.242138,2254.000000,gbp,1975,2,13,0.161413,0.103075,NaN,2.3901,NaN,44057.292033
1975-02-13,563,"Postiglione, Vincenzo II",Violin,Naples,Sotheby's,1975-02-13,219,4421.000000,17948.156818,2194.000000,gbp,1975,2,13,0.161413,0.103075,NaN,2.3901,NaN,42897.889612
1975-02-13,2522,"Dominichino, Giuseppe",Violin,Verona,Sotheby's,1975-02-13,213,2151.000000,8731.535749,1068.000000,gbp,1975,2,13,0.161413,0.103075,NaN,2.3901,NaN,20869.243595
1975-05-01,1497,"Gagliano, Nicolò",Violin,Naples,Phillip's,1975-05-01,-,9862.000000,36929.958930,NaN,gbp,1975,5,1,0.163565,0.113729,NaN,2.3491,NaN,86752.166523
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-21,806,"Vuillaume, Nicolas François",Cello,Brussels,NaN,2025-11-21,NaN,30735.000000,23486.000000,26723.345196,eur,2025,11,21,0.996526,1.000000,0.998378,1.3085,1.1515,30771.931993
2025-11-21,1638,"Kunze, Wilhelm Paul",Violin,The Hague,NaN,2025-11-21,NaN,2719.000000,2078.000000,2363.834133,eur,2025,11,21,0.996526,1.000000,0.998378,1.3085,1.1515,2721.955004
2025-11-21,1213,"Prager, Gustav Adolf",Violin Bow,NaN,NaN,2025-11-21,NaN,755.000000,577.000000,656.064134,eur,2025,11,21,0.996526,1.000000,0.998378,1.3085,1.1515,755.457851


## Build Financial Market Climate Indicator

In [9]:
########## SP500 Index ########
# returns
# consider minimally monthly returns as art markets are delayed and have frictions and you cannot purchase instantan.
SP['SP500_30d_ret'] = SP['SP500_real'].pct_change(30)
SP['SP500_90d_ret'] = SP['SP500_real'].pct_change(90)
SP['SP500_252d_ret'] = SP['SP500_real'].pct_change(252)

# annualised volatilities on SP500
daily_ret = SP['SP500_real'].pct_change()
SP['SP500_vol_30d']  = daily_ret.rolling(30).std() * np.sqrt(252)
SP['SP500_vol_90d']  = daily_ret.rolling(90).std() * np.sqrt(252)
SP['SP500_vol_252d'] = daily_ret.rolling(252).std() * np.sqrt(252)

# Trend Indicator based on moving averages / smoothing over different time horizons
SP['SP500_ma50']  = SP['SP500_real'].rolling(50).mean()
SP['SP500_ma200'] = SP['SP500_real'].rolling(200).mean()

SP['SP500_trend_ratio'] = SP['SP500_ma50'] / SP['SP500_ma200'] # >1 means bullish


########## FX ########
# FX indicators (based on GBP as it exists for full history unlike Euro)
FX_GBP['gbp_90d_change'] = FX_GBP['gbp_usd'].pct_change(90)
FX_GBP['gbp_vol_30d'] = FX_GBP['gbp_usd'].pct_change().rolling(30).std()
#FX_GBP['gbp_vol_30d'] = 0.5 * (FX_GBP['gbp_vol_30d'].ffill() + FX_GBP['gbp_vol_30d'].bfill()) # linearly interpolate
FX_GBP['gbp_vol_30d'] = FX_GBP['gbp_vol_30d'].ffill()# linearly interpolate

########## Yield ########
# Rate indicators based on 10Y US T-Note Yield level (in percent)
US10['10y_yield_90d_change'] = US10['10y_yield_pc'].diff(90) / 100 # get rid of percent

In [10]:
macro = SP.join(FX_GBP).join(US10)
macro.drop(columns=['Year', 'Month', 'Day'], inplace=True)

In [11]:
# z-score normalisation to bring all indicators to the same scale
macro['z_ret'] = z(macro['SP500_252d_ret'])
macro['z_vol'] = z(macro['SP500_vol_30d'])
macro['z_rate'] = z(macro['10y_yield_pc'])
macro['z_drate'] = z(macro['10y_yield_90d_change'])
#macro['z_fx'] = z(macro['gbp_vol_30d'])  # fx vola column
macro['z_fx'] = ((macro['gbp_vol_30d']- macro['gbp_vol_30d'].ffill().rolling(252).mean()) / macro['gbp_vol_30d'].ffill().rolling(252).std())

In [15]:
macro['z_fx'].dropna()

Date
1976-02-11   -1.689573
1976-02-12   -1.661481
1976-02-13   -1.644290
1976-02-17   -1.814750
1976-02-18   -1.829827
                ...   
2026-02-12   -0.041639
2026-02-13   -0.026546
2026-02-17    0.308105
2026-02-18    0.328092
2026-02-19    0.453303
Name: z_fx, Length: 12386, dtype: float64

Below, we built a market climate indicator (MCI), which ought to capture the current state of the market.
- high MCI: strong markets, liquidity, risk appetite.
- Low MCI: stressed markets

This indicator is good because it normalizes the scales of all underlying features, it adapts across regimes via rolling-window statistics, avoids multicollinearity by combining many highly correlated features into a single indicator/state-variable. In this way and through the smoothing it reduces noise, which implies less overfitting and hopefully will improve out-of-sample prediction.

In [12]:
# construct market climate indicator as single state variable (to reduce noise, multicollinearity, etc.)
# high MCI: strong markets, liquidity, risk appetite. Low MCI: stressed markets
macro['MCI'] = macro['z_ret'] - macro['z_vol'] - macro['z_rate'] - macro['z_drate'] - macro['z_fx']
# smooth indicator to not overfit noise
macro['MCI'] = macro['MCI'].rolling(30, min_periods=1).mean()

In [18]:
macro['1976-02-11':'2025'].head(30)#.MCI.plot()

,SP500_real,SP500_30d_ret,SP500_90d_ret,SP500_252d_ret,SP500_vol_30d,SP500_vol_90d,SP500_vol_252d,SP500_ma50,SP500_ma200,SP500_trend_ratio,...,gbp_90d_change,gbp_vol_30d,10y_yield_pc,10y_yield_90d_change,z_ret,z_vol,z_rate,z_drate,z_fx,MCI
Date,,,,,,,,,,,,,,,,,,,,,
1976-02-11,587.377114,0.116500,0.153516,0.186317,0.132730,0.128766,0.149289,549.878118,538.110388,1.021869,...,-0.004467,0.001647,7.850,-0.00450,0.864990,-0.686595,-0.627910,-1.265361,-1.689573,5.134430
1976-02-12,584.346111,0.105566,0.135279,0.164316,0.135150,0.128309,0.148839,551.094855,538.348328,1.023677,...,-0.008322,0.001657,7.835,-0.00395,0.726099,-0.578610,-0.691961,-1.147304,-1.661481,4.969943
1976-02-13,580.965345,0.092550,0.130142,0.150620,0.137454,0.128801,0.148879,552.446748,538.560044,1.025785,...,-0.008325,0.001658,7.820,-0.00330,0.637187,-0.475541,-0.760289,-1.009151,-1.644290,4.822114
1976-02-17,577.351462,0.066051,0.108169,0.151516,0.131792,0.127780,0.148841,553.698233,538.719453,1.027804,...,-0.012110,0.001479,7.780,-0.00410,0.636474,-0.709490,-0.923906,-1.188493,-1.814750,4.934864
1976-02-18,582.014547,0.063745,0.111684,0.153547,0.130774,0.128153,0.148920,555.162532,538.875889,1.030223,...,-0.016569,0.001445,7.790,-0.00400,0.642690,-0.746340,-0.908169,-1.171385,-1.829827,5.007573
1976-02-19,591.107644,0.075535,0.131100,0.160597,0.136405,0.130293,0.149415,556.779391,539.121809,1.032752,...,-0.018008,0.001446,7.800,-0.00270,0.679620,-0.500858,-0.890368,-0.891023,-1.809564,4.968217
1976-02-20,595.129547,0.075640,0.122884,0.162695,0.136432,0.128823,0.149479,558.449729,539.374388,1.035366,...,-0.011333,0.001423,7.770,-0.00300,0.686329,-0.494726,-1.028860,-0.964088,-1.813066,4.970910
1976-02-23,592.273404,0.066307,0.119748,0.173880,0.137959,0.129112,0.148828,559.971523,539.598014,1.037757,...,-0.015565,0.001417,7.720,-0.00350,0.749182,-0.424454,-1.238675,-1.076926,-1.799252,5.010607
1976-02-24,594.721526,0.055375,0.125007,0.207041,0.133072,0.129160,0.146880,561.575097,539.804228,1.040331,...,-0.014981,0.001215,7.690,-0.00360,0.948781,-0.627707,-1.368893,-1.099510,-1.983095,5.123649


In [18]:
US10

,10y_yield_pc,10y_yield_90d_change
observation_date,,
1962-01-02,4.060,NaN
1962-01-03,4.030,NaN
1962-01-04,3.990,NaN
1962-01-05,4.020,NaN
1962-01-08,4.030,NaN
...,...,...
2026-02-13,4.040,-0.00010
2026-02-16,4.045,0.00005
2026-02-17,4.050,0.00020


In [19]:
start_idx = macro.dropna().index[0]
# exclude z-score features as they are already in the 'sales' data set 
sales = attach_timeseries_features(sales[start_idx:], macro[start_idx:],
                                  cols = [c for c in macro.columns if c not in ['z_ret', 'z_vol', 'z_rate', 'z_drate', 'z_fx']])

In [21]:
sales2 = attach_timeseries_features(sales, Gold)

In [29]:
sales.columns

Index(['maker_id', 'maker_name', 'type', 'city_maker', 'auction_house',
       'sale_date', 'lot', 'usd', 'gbp', 'eur', 'bold_currency', 'Year',
       'Month', 'Day', 'cpi_usd', 'cpi_gbp', 'cpi_eur', 'gbp_usd_x', 'eur_usd',
       'price_usd_real', 'SP500_real', 'SP500_30d_ret', 'SP500_90d_ret',
       'SP500_252d_ret', 'SP500_vol_30d', 'SP500_vol_90d', 'SP500_vol_252d',
       'SP500_ma50', 'SP500_ma200', 'SP500_trend_ratio', 'gbp_usd_y',
       'gbp_90d_change', 'gbp_vol_30d', '10y_yield_pc', '10y_yield_90d_change',
       'MCI', 'real_price_gold'],
      dtype='object')

In [30]:
sales.to_csv('price_adj_all_w_econ_feeatures.csv')